# 📑 Notebook 21: GenAI RAG Clinical Protocol Auditor & SLA Governance
## Retrieval-Augmented Generation (RAG) over Hospital Standard Operating Procedures

**Author:** CareFlow Clinical Governance & GenAI Engineering Team  
**Pillars:** Gen AI, RAG Knowledge Systems, Clinical SLA Audit & Process Compliance  

---

### Overview
Clinical workflows in modern hospitals are governed by rigorous **Standard Operating Procedures (SOPs)**—such as Sepsis bundles, STEMI door-to-balloon targets, and triage consultation windows. However, manual audits of process compliance are infrequent and retrospective.

In this notebook, we implement an **Automated RAG Clinical Protocol Auditor** that:
1. Embeds and indexes official hospital clinical policies.
2. Implements a semantic retrieval system querying relevant operational mandates in real time.
3. Compares actual telemetry wait times from the CareFlow lakehouse against SOP thresholds.
4. Synthesizes cited, audit-grade generative compliance reports.

In [ ]:
# 1. Load Dependencies & RAG Engine
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from careflow_analytics.genai_engine import (
    audit_clinical_protocol_rag,
    HOSPITAL_CLINICAL_PROTOCOLS
)

print(f"[OK] Loaded {len(HOSPITAL_CLINICAL_PROTOCOLS)} hospital clinical SOPs into RAG vector-space.")
pd.DataFrame(HOSPITAL_CLINICAL_PROTOCOLS)[['id', 'title', 'category']]

### 2. Querying Clinical Protocols via Semantic RAG
We test the RAG engine with queries regarding door-to-doctor time limits and sepsis resuscitation guidelines.

In [ ]:
query_1 = "What is the maximum allowed door to doctor time for emergent acuity 2 patients?"
res_1 = audit_clinical_protocol_rag(query_1)

print(f"Matched SOP: {res_1['matched_protocol_id']} - {res_1['matched_protocol_title']}")
print(f"Confidence: {res_1['confidence']}\n")
print(res_1['rag_response'])

### 3. Evaluating Sepsis Protocol Retrieval

In [ ]:
query_2 = "How quickly must blood cultures and IV antibiotics be administered in suspected sepsis?"
res_2 = audit_clinical_protocol_rag(query_2)

print(f"Matched SOP: {res_2['matched_protocol_id']} - {res_2['matched_protocol_title']}")
print(res_2['rag_response'])

### 4. Automated Compliance Auditing on Lakehouse Telemetry
We load `mart_sla_compliance.parquet` and benchmark real emergency cases against the 30-minute Door-to-Doctor SLA mandate.

In [ ]:
DATA_DIR = '../careflow_analytics/output'
sla_path = os.path.join(DATA_DIR, 'mart_sla_compliance.parquet')

if os.path.exists(sla_path):
    sla_df = pd.read_parquet(sla_path)
    print(f"[OK] Analyzed {len(sla_df):,} cases in SLA compliance mart.")
    
    breach_rate = (sla_df['door_to_doc_breach'].sum() / len(sla_df)) * 100
    print(f"Door-to-Doctor SLA Breach Rate: {breach_rate:.1f}%")
    print(f"Compliant Cases: {(~sla_df['door_to_doc_breach']).sum():,} | Breached: {sla_df['door_to_doc_breach'].sum():,}")
else:
    print("[NOTE] Run dbt pipeline to inspect parquet mart.")

### Conclusion & Institutional Significance
- Integrating RAG knowledge retrieval with lakehouse event streams bridges the gap between written policy and clinical reality.
- Hospital quality committees can deploy this automated auditor for real-time compliance alerting and accreditation governance.